# Europe's Macroeconomic Pulse — A Data-Driven Overview

> **TL;DR** — I built an open-source pipeline that pulls fresh data from the
> [Eurostat API](https://ec.europa.eu/eurostat), transforms it with
> **dbt + DuckDB**, and visualises the results below.
> All code lives in a single repo you can run with three commands.

What follows is a set of charts that answer one question:
*How are EU economies really doing across growth, jobs, prices, and debt?*

In [11]:
import duckdb
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import pandas as pd

# ── Shared chart theme ──────────────────────────────────────────────
TEMPLATE = dict(
    layout=go.Layout(
        font=dict(family="Inter, Arial, sans-serif", size=13, color="#2d2d2d"),
        title=dict(font=dict(size=20, color="#1a1a2e"), x=0.01, xanchor="left"),
        plot_bgcolor="#fafafa",
        paper_bgcolor="#ffffff",
        colorway=["#264653","#2a9d8f","#e9c46a","#f4a261","#e76f51","#6a4c93"],
        margin=dict(l=60, r=30, t=70, b=50),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    )
)
pio.templates["substack"] = go.layout.Template(**TEMPLATE)
pio.templates.default = "plotly_white+substack"

# ── Connect to warehouse ────────────────────────────────────────────
db = duckdb.connect("../data/warehouse.duckdb", read_only=True)

print("Gold layer tables:")
db.sql("SELECT table_name FROM information_schema.tables WHERE table_schema = 'gold' ORDER BY 1").show()

Gold layer tables:
┌─────────────────────────┐
│       table_name        │
│         varchar         │
├─────────────────────────┤
│ gold_dim_countries      │
│ gold_dim_customers      │
│ gold_dim_dates          │
│ gold_dim_orders         │
│ gold_dim_products       │
│ gold_fct_gdp            │
│ gold_fct_gov_finance    │
│ gold_fct_inflation      │
│ gold_fct_macro_overview │
│ gold_fct_sales          │
│ gold_fct_unemployment   │
├─────────────────────────┤
│         11 rows         │
└─────────────────────────┘



## 1 — GDP Growth: Who's Growing and Who's Stalling?

Real GDP growth (chain-linked volumes) strips out inflation to show the
*actual* expansion or contraction of an economy. The heatmap below makes
it easy to spot the **COVID cliff in 2020** and the uneven recovery that
followed.

In [7]:
df_gdp = db.sql("""
    select g.country_code, c.country_name, g.year, g.real_gdp_growth_pct
    from gold.gold_fct_gdp g
    join gold.gold_dim_countries c using (country_code)
    where c.is_eu_member
      and cast(g.year as integer) >= 2015
      and g.real_gdp_growth_pct is not null
    order by country_name, year
""").df()

pivot = df_gdp.pivot(index="country_name", columns="year", values="real_gdp_growth_pct")

fig = px.imshow(
    pivot,
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    title="Real GDP Growth (%) — EU Member States since 2015",
    labels=dict(x="Year", y="", color="Growth %"),
    aspect="auto",
    text_auto=".1f",
)
fig.update_layout(height=600, coloraxis_colorbar=dict(title="% YoY"))
fig.show()

## 2 — Unemployment: The Labour-Market Divergence

Unemployment rates tell a story of structural gaps.
Southern Europe (Spain, Italy) consistently runs higher rates than the
north-western core (Germany, Netherlands). The chart below tracks
six of the largest EU economies month by month.

In [8]:
COUNTRIES = ("DE", "FR", "ES", "IT", "NL", "PL")
LABELS = {"DE":"Germany","FR":"France","ES":"Spain","IT":"Italy","NL":"Netherlands","PL":"Poland"}

df_unemp = db.sql(f"""
    select country_code, month, unemployment_rate_total
    from gold.gold_fct_unemployment
    where country_code in {COUNTRIES}
      and month >= '2015-01'
    order by country_code, month
""").df()
df_unemp["country"] = df_unemp["country_code"].map(LABELS)

fig = px.line(
    df_unemp, x="month", y="unemployment_rate_total", color="country",
    title="Monthly Unemployment Rate (%) — Selected EU Economies",
    labels={"month": "", "unemployment_rate_total": "Unemployment %", "country": ""},
)
fig.update_layout(height=450, hovermode="x unified")
fig.update_traces(line=dict(width=2.2))
fig.show()

## 3 — Inflation: The Energy Shock and Its Aftermath

The 2022 energy crisis sent headline HICP inflation soaring across Europe.
The area chart below decomposes inflation into its key categories for the
Euro area aggregate (EA), highlighting how **energy prices** were the dominant
driver — and how quickly they normalised relative to food and housing.

In [12]:
df_infl = db.sql("""
    select month,
           headline_inflation_pct,
           food_inflation_pct,
           housing_inflation_pct,
           transport_inflation_pct,
           energy_inflation_pct
    from gold.gold_fct_inflation
    where country_code = 'EA'
      and month >= '2018-01'
    order by month
""").df()

fig = go.Figure()
for col, name, color in [
    ("energy_inflation_pct",    "Energy",    "#e76f51"),
    ("food_inflation_pct",      "Food",      "#e9c46a"),
    ("housing_inflation_pct",   "Housing",   "#2a9d8f"),
    ("transport_inflation_pct", "Transport", "#264653"),
]:
    fig.add_trace(go.Scatter(
        x=df_infl["month"], y=df_infl[col], name=name,
        mode="lines", stackgroup="one",
        line=dict(width=0.5, color=color),
    ))
fig.add_trace(go.Scatter(
    x=df_infl["month"], y=df_infl["headline_inflation_pct"],
    name="Headline HICP", mode="lines",
    line=dict(color="#1a1a2e", width=2.5, dash="dot"),
))
fig.update_layout(
    title="HICP Inflation Breakdown — Euro Area",
    yaxis_title="YoY %", xaxis_title="",
    height=480, hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
)
fig.show()

## 4 — Government Debt: The Fiscal Landscape

The Maastricht criteria set 60 % of GDP as the debt ceiling — a target
most large EU members overshoot. The horizontal bar chart ranks the
**latest year** of reported gross debt for every EU country in the dataset.

In [20]:
df_debt = db.sql("""
    with latest as (
        select country_code, max(year) as latest_year
        from gold.gold_fct_gov_finance
        where gross_debt_pct_gdp is not null
        group by country_code
    )
    select g.country_code, c.country_name, g.year, g.gross_debt_pct_gdp
    from gold.gold_fct_gov_finance g
    join latest l on g.country_code = l.country_code and g.year = l.latest_year
    join gold.gold_dim_countries c on g.country_code = c.country_code
    where c.is_eu_member
    order by g.gross_debt_pct_gdp desc
""").df()

fig = px.bar(
    df_debt, x="gross_debt_pct_gdp", y="country_name", orientation="h",
    title=f"Government Gross Debt (% of GDP) — Latest Year ({int(df_debt['year'].iloc[0])})",
    labels={"gross_debt_pct_gdp": "% of GDP", "country_name": ""},
    text_auto=".0f",
)
fig.add_vline(x=60, line_dash="dash", line_color="#e76f51", annotation_text="Maastricht 60 %")
fig.update_layout(height=600, showlegend=False)
fig.update_traces(marker_color=df_debt["gross_debt_pct_gdp"].apply(
    lambda v: "#e76f51" if v > 100 else "#f4a261" if v > 60 else "#2a9d8f"
))
fig.show()

## 5 — The Phillips Curve: Inflation vs. Unemployment

The classic Phillips curve suggests an inverse relationship between
unemployment and inflation. Let's plot the latest monthly snapshot for
every EU country to see if the theory holds in the current economic climate.

In [14]:
df_phillips = db.sql("""
    with latest_month as (
        select max(period) as m from gold.gold_fct_macro_overview
        where headline_inflation_pct is not null
          and unemployment_rate_total is not null
    )
    select o.country_code, c.country_name,
           o.unemployment_rate_total,
           o.headline_inflation_pct
    from gold.gold_fct_macro_overview o
    join latest_month lm on o.period = lm.m
    join gold.gold_dim_countries c using (country_code)
    where c.is_eu_member
      and o.unemployment_rate_total is not null
      and o.headline_inflation_pct is not null
""").df()

fig = px.scatter(
    df_phillips, x="unemployment_rate_total", y="headline_inflation_pct",
    text="country_code", size_max=14,
    title="Phillips Curve Snapshot — EU Countries (Latest Month)",
    labels={"unemployment_rate_total": "Unemployment Rate (%)",
            "headline_inflation_pct": "Headline Inflation (%)"},
)
fig.update_traces(
    textposition="top center", marker=dict(size=12, color="#264653", opacity=0.8)
)
fig.add_hline(y=2, line_dash="dash", line_color="#e76f51",
              annotation_text="ECB 2 % target", annotation_position="bottom right")
fig.update_layout(height=500, showlegend=False)
fig.show()

## 6 — Sentiment & Rates: Consumer Confidence vs. Interest Rates

Consumer confidence is a leading indicator — when households feel pessimistic,
spending contracts. Meanwhile, central-bank interest rates respond to inflation.
The dual-axis chart overlays both for the Euro area to show their interplay.

**How to read this chart:**
- **Consumer Confidence Index** (blue line, left axis): Ranges from roughly -40 to +30. Values above 0 indicate optimism; below 0 indicates pessimism. The index is *normally negative* — a reading of -10 is typical in stable times. Steep drops signal consumers bracing for trouble.
- **3-Month Interest Rate** (orange line, right axis): The short-term money-market rate, closely tracking ECB policy. The ECB held rates at or below 0 % from 2014–2022 (the "negative-rate era"), then hiked aggressively to fight the post-2022 inflation surge.

In [26]:
from plotly.subplots import make_subplots

# EA aggregate has no confidence data — average the Big-4 economies instead
df_conf = db.sql("""
    select period,
           avg(consumer_confidence) as avg_confidence
    from gold.gold_fct_macro_overview
    where country_code in ('DE','FR','IT','ES')
      and period >= '2015-01'
      and consumer_confidence is not null
    group by period
    order by period
""").df()

df_rates = db.sql("""
    select period, interest_rate_3m
    from gold.gold_fct_macro_overview
    where country_code = 'EA'
      and period >= '2015-01'
      and interest_rate_3m is not null
    order by period
""").df()

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(
    x=df_conf["period"], y=df_conf["avg_confidence"],
    name="Consumer Confidence (DE/FR/IT/ES avg)", mode="lines",
    line=dict(color="#264653", width=2),
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=df_rates["period"], y=df_rates["interest_rate_3m"],
    name="3-Month Rate (Euro Area)", mode="lines",
    line=dict(color="#e76f51", width=2),
), secondary_y=True)

fig.update_layout(
    title="Consumer Confidence vs. Short-Term Interest Rate",
    height=450, hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
)
fig.update_yaxes(title_text="Confidence Index", secondary_y=False)
fig.update_yaxes(title_text="3M Rate (%)", secondary_y=True)
fig.show()

## Key Takeaways

| Theme | Insight |
|---|---|
| **Growth** | The 2020 COVID shock is the dominant feature; recovery strength varies widely across EU members |
| **Jobs** | Southern Europe still carries structurally higher unemployment — the gap hasn't closed |
| **Prices** | The 2022 energy price spike drove most of the inflation surge; core categories were far more muted |
| **Debt** | Greece, Italy, and France sit well above the 60 % Maastricht threshold |
| **Sentiment** | Consumer confidence collapsed alongside the energy crisis and has been slow to recover |

---

### How This Was Built

```
Eurostat JSON API  →  Python ingestion  →  Parquet  →  dbt (bronze → silver → gold)  →  DuckDB  →  Plotly
```

The entire pipeline runs in **one command**: `uv run pipeline eurostat`.
All source code is open — feel free to fork, extend, or connect your own data sources.

---
*Data source: [Eurostat](https://ec.europa.eu/eurostat) — EU statistical office.
Extracted and transformed with dbt + DuckDB. Visualised with Plotly.*

In [ ]:
db.close()
print("Connection closed.")